In [1]:
# import os
# os.environ['OPENAI_API_KEY'] = 'set-your-key-in-shell-or-.env'  # do not hardcode


In [2]:
import cv2
import torch
import pyttsx3
from datetime import datetime
import numpy as np
import requests
from IPython.display import display, Audio
import openai
import time
from PIL import Image
import sys
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import pygame
import os

pygame 2.5.2 (SDL 2.28.3, Python 3.9.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
# YOLOv7
sys.path.insert(0, './yolov7')
from models.experimental import attempt_load  # Import the function to load the model
from utils.general import non_max_suppression  # Import the utility function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = attempt_load('C:/Users/Tesisti/Desktop/Yuchen 2023-2026/06 GPT+YOLOV7/yolov7/best.pt', map_location=device)
model.eval()  # Set the model to evaluation mode
yolov7_label_order = ['Allen key', 'Hex cap screw (big)', 'Drive shaft companion flange', 'Nuts', 'Pump housing', 'S12', 'S6', 'S9', 'Washers', 'Wrench size 13']

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
IDetect.fuse


In [5]:
# Assembly data loading
assembly_tasks_path = 'assembly_tasks.xlsx'
assembly_tasks_df = pd.read_excel(assembly_tasks_path)
index_total = assembly_tasks_df.dropna(how='all').shape[0] - 1

steps_classes = assembly_tasks_df['step'].dropna().to_dict()
components_classes = assembly_tasks_df['components'].dropna().to_dict()
tools_classes = assembly_tasks_df['tools'].dropna().to_dict()
operations_classes = assembly_tasks_df['operations'].dropna().to_dict()
time_classes = assembly_tasks_df['time'].dropna().to_dict()
lastorder_classes = assembly_tasks_df['lastorder'].dropna().to_dict()
print(steps_classes)
print(components_classes)
print(tools_classes)

{0: 'S6', 1: 'S9', 2: 'S12'}
{0: 'Hex cap screw (big),C26', 1: 'Drive shaft companion flange,C8', 2: 'Pump housing,C2;Washers,C17;Nuts,C18'}
{0: 'None', 1: 'Allen key,T5', 2: 'Wrench size 13,T6'}


In [6]:
# Error information
error_templates_path = 'error_templates.xlsx'
error_templates_df = pd.read_excel(error_templates_path)
error_templates = dict(zip(error_templates_df['error_type'], error_templates_df['template']))
print(error_templates)

{'components': '\n# Error Log\nDate and Time: {now}\nReported by: {operator_name}\nLocation: {location}\n\n## Error Description\nStep: {step}\nTask: {task}\nExpected Component: {expected}\nActual Component: {actual}\n\n## Error Details\nDescription:\nThe operator mistakenly picked the {actual} instead of the {expected} during step {step} of the assembly process.\n\n## Corrective Actions\nImmediate Action Taken:\n- Replace the {actual} with the {expected}\n', 'tools': '\n# Error Log\nDate and Time: {now}\nReported by: {operator_name}\nLocation: {location}\n\n## Error Description\nStep: {step}\nTask: {task}\nExpected Tool: {expected}\nActual Tool: {actual}\n\n## Error Details\nDescription:\nThe operator mistakenly used the {actual} instead of the {expected} during step {step} of the assembly process.\n\n## Corrective Actions\nImmediate Action Taken:\n- Replace the {actual} with the {expected}\n', 'steps': '\n# Error Log\nDate and Time: {now}\nReported by: {operator_name}\nLocation: {loca

In [7]:
# Image display 
def show_image(frame):
    try:
        frame_rgb = cv1.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.imshow(frame_rgb)
        plt.axis('off')
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        plt.title(f'Image_{timestamp}')
        plt.show()  # Ensure the image is displayed when the function is called
    except Exception as e:
        print(f"Failed to display image: {e}")

In [8]:
# Image preprocessing
def preprocess_image(frame):
    try:
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        img = np.expand_dims(img, 0)
        img = torch.from_numpy(img).float()
        return img
    except Exception as e:
        print(f"Error during image preprocessing: {e}")
        return None

In [9]:
# Image preprocessing cancel
def postprocess_image(tensor):
    if isinstance(tensor, torch.Tensor):
        tensor = tensor.cpu()
        image_np = tensor.numpy()
        image_np = image_np[0]
        if image_np.shape[0] < image_np.shape[2]:
            image_np = image_np.transpose(1, 2, 0)
        if image_np.max() <= 1.0:
            image_np = (image_np * 255).astype(np.uint8)
        return image_np
    else:
        print("传入的数据不是 torch.Tensor 类型")
        return None

In [10]:
# Image capturing
def capture_image():
    cap = cv2.VideoCapture(0)
    ret, frame = cap.read()
    cap.release()
    if ret:
        # show_image(frame)
        img = preprocess_image(frame)
        return img
    else:
        return None

In [21]:
# Image YOLOv7 detection
def detect_items(image):
    frame = image.squeeze().numpy().transpose(1, 2, 0) * 255  # 转换图像为适合显示的格式
    # frame = frame.astype(np.uint8)

    with torch.no_grad():
        predictions = model(image)  # 使用模型进行预测
        predictions = predictions[0] if isinstance(predictions, tuple) else predictions
        predictions = non_max_suppression(predictions, conf_thres=0.6, iou_thres=0.45)

    detected_items = []  # 初始化检测到的物体列表

    # 检查predictions是否存在且不为空
    if predictions is not None and len(predictions) > 0 and len(predictions[0]) > 0:
        detected_items = predictions[0]  # 提取检测到的物体
        print(f"YOLOv7 detections found: {len(detected_items)}")  # 打印检测到的物体数量
    else:
        print("No detections found.")

    return detected_items

In [12]:
# detected items classification
def classify_detections(detected):
    classified_items = {
        'components': [],
        'tools': [],
        'steps': []
    }
    for detection in detected:
        x1, y1, x2, y2, conf, cls = detection[:6]
        if conf > 0.5:
            label = yolov7_label_order[int(cls)]
            print('label:' + label)
            if any(label == step for step in steps_classes.values()):
                classified_items['steps'].append(label)
            if any(label in component for component in components_classes.values()):
                classified_items['components'].append(label)
            if any(label in tool for tool in tools_classes.values()):
                classified_items['tools'].append(label)
    return classified_items

In [13]:
# Error validation and handle
def error_validation(classified, step_details, operator_name, location, item_type, task):
    details = {
        'now': datetime.now().strftime("%Y-%m-%d %H:%M"),
        'operator_name': operator_name,
        'location': location,
        'expected': step_details[item_type],
        'actual': classified[item_type],
        'step': step_details['step'],
        'task': task
    }
    
    error_result = {
        'type': item_type,
        'specific': details['actual'],
        'result': True
    }
    
    if set(details['actual']).issubset(set(details['expected'])):
        error_result['result'] = True
        error_log = None
    else:
        error_result['result'] = False
        error_log = error_templates[item_type].format(**details)  # Ensure error_templates is defined with proper templates.
        
    errorLog_dir = "C:/Users/Tesisti/Desktop/Yuchen 2023-2026/06 GPT+YOLOV7/Results_ErrorLog"
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    errorLog_filename = f"{errorLog_dir}/text{current_time}.txt"
    with open(errorLog_filename, "w") as text_file:
        text_file.write(error_log)
    print(f"Error log saved.")
    
    return error_result, error_log

In [14]:
# Label with different colors according to different situation
def annotate_image_with_labels(image, detected_items, classified_items, item_type, error_result):
    # Define colors for different scenarios
    correct_color = (0, 255, 0)  # Green for "True"
    wrong_color = (255, 0, 0)    # Red for "False"
    neutral_color = (128, 128, 128)  # Gray for other categories

    # Process the image once
    processed_image = postprocess_image(image)

    for detection in detected_items:
        x1, y1, x2, y2, conf, cls = detection[:6]
        detection_label = yolov7_label_order[int(cls)]  # Convert class ID to readable label
        
        # Determine the color based on classification and result
        if detection_label in classified_items[item_type]:
            color = correct_color if error_result['result'] else wrong_color
        else:
            color = neutral_color
        
        # Draw rectangle and label on the same processed image
        cv2.rectangle(processed_image, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(processed_image, detection_label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Save and optionally display the annotated image
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    annotated_image_path = f'./Results_Image/annotated_{item_type}_{timestamp}.jpg'
    cv2.imwrite(annotated_image_path, processed_image)
    print(f"Annotated image saved.")

    return processed_image

In [15]:
# Load gpt prompt
def generate_prompt(step_details, error_description):
    gpt_prompt_path = 'gpt_prompt.xlsx'
    gpt_prompt_df = pd.read_excel(gpt_prompt_path)
    prompt_template = gpt_prompt_df['prompt'].iloc[0]
    prompt = prompt_template.format(step_details=step_details, error_description=error_description)
    return prompt

In [16]:
def GPT_TTS(prompt):
    # Check GPT API key
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise ValueError("API key is not set. Please ensure the 'OPENAI_API_KEY' environment variable is defined.")
    
    # Step1: Text generation
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.choices[0].message.content.strip()
    # print("Generated text:", text)
    
    # Text result save
    results_text_dir = "C:/Users/Tesisti/Desktop/Yuchen 2023-2024/06 GPT+YOLOV7/Results_Text"
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    text_filename = f"{results_text_dir}/text{current_time}.txt"
    with open(text_filename, "w") as text_file:
        text_file.write(text)
    print(f"Results text saved.")

    # Step2: Speech generation
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
    }
    data = {
        "model": "tts-1",
        "input": text,
        "voice": "alloy"
    }
    response = requests.post('https://api.openai.com/v1/audio/speech', headers=headers, json=data)
    if response.status_code == 200:
        results_speech_dir = "C:/Users/Tesisti/Desktop/Yuchen 2023-2026/06 GPT+YOLOV7/Results_Speech"
        speech_filename = f"{results_speech_dir}/speech{current_time}.mp3"
        with open(speech_filename, 'wb') as file:
            file.write(response.content)
        print("Error instruction speech saved.")
        # display(Audio(speech_filename, autoplay=True))
    else:
        print("Failed to generate speech:", response.status_code, response.text)
    return speech_filename

In [17]:
# Text to speech
def text_to_speech(text):
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    voice_path = f'./TTS_Speech/voice_{timestamp}.mp3'
    try:
        engine = pyttsx3.init()
        rate = engine.getProperty('rate')  # Obatain current speed
        engine.setProperty('rate', rate - 80)  # Modify speed
        volume = engine.getProperty('volume')  # Obatain current volume
        engine.setProperty('volume', volume + 0.25)  # Modify volume
        voices = engine.getProperty('voices')  # Obatain aviable voice
        engine.setProperty('voice', voices[0].id)  # Modify voice（0-male，1-female）
        engine.say(text)
        engine.save_to_file(text, voice_path)
        engine.runAndWait()
    except Exception as e:
        print(f"Error with text-to-speech: {e}")
        # print(text)

In [18]:
def play_audio(filename):
    pygame.mixer.init()
    pygame.mixer.music.load(filename)
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        pygame.time.Clock().tick(10)

In [19]:
# main
def main():
    operator_name = input("Please enter the operator's name: ")
    location = "Assembly Line 3, Workstation 7"
    for index in range(index_total):
        # Obtain all the information for this step
        step_details = {
            'components': components_classes[index],
            'tools': tools_classes[index],
            'operations': operations_classes[index],
            'time': time_classes[index],
            'step': steps_classes[index],
            'steps': steps_classes[index]
        }
        # print(step_details)

        for item_type in ['components', 'tools', 'steps']:
            print(f"——————————————Start step{index+1}.{item_type} ——————————————")
            if step_details[item_type] != "None":
                retry = True
                retry_count = 0
                while retry and retry_count < 3:
                    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
                    file_name = f"{item_type}_{timestamp}.mp4"
                    if item_type in ['components', 'tools']:
                        Task = f"Please prepare the following {item_type}: {step_details[item_type]}."
                        print(Task)
                        text_to_speech(Task)
                        sleep_time = 3
                    else:
                        Task = f"Please follow the instructions to complete the step: {step_details['operations']}."
                        text_to_speech(Task)
                        sleep_time = step_details['time']

                    time.sleep(sleep_time)
                    image = capture_image()
                    detected_items = detect_items(image)
                    # print(detected_items)
                    classified_items = classify_detections(detected_items)
                    # print(classified_items)
                    error_result, error_log = error_validation(classified_items, step_details, operator_name, location, item_type, Task)
                    # print(error_log)
                    image = annotate_image_with_labels(image, detected_items, classified_items, item_type, error_result)

                    if error_result['result'] == False:
                        if retry_count < 2:
                            prompt = generate_prompt(step_details, error_log)
                            speech_filename = GPT_TTS(prompt)
                            play_audio(speech_filename)
                            print(f"——————————————Redo step{index+1}.{item_type} ——————————————")
                            retry_count += 1
                        else:
                            print("Failed to correct the error after multiple attempts.")
                            retry = False
                    else:
                        text_to_speech("You did it correctly. Now let's proceed to the next step.")
                        print("You did it correctly. Now let's proceed to the next step.")
                        print(f"——————————————End step{index+1}.{item_type}——————————————")
                        retry = False

In [22]:
if __name__ == '__main__':
    main()

Please enter the operator's name: yuchen
——————————————Start step1.components ——————————————
Please prepare the following components: Hex cap screw (big),C26.


AttributeError: 'NoneType' object has no attribute 'squeeze'